In [1]:
# Auto-switch to repo root so relative paths (configs/, scripts/, data/) work
from pathlib import Path
import os

_cwd = Path.cwd()
_root = _cwd
while _root != _root.parent:
    if (_root / "configs" / "config.yaml").exists():
        break
    _root = _root.parent
os.chdir(_root)
print(f"Repo root: {_root}")


Repo root: c:\Users\karki\Downloads\main_openworld\openworld


# 02 — Train the base-class Faster R-CNN (Phase 2)

Trains Faster R-CNN (ResNet-50 + FPN, ImageNet backbone only, no detector pretraining)
on the 15 base classes.

Start with the smoke run (200 images, 1 epoch) to verify the pipeline end-to-end on CPU,
then run full training (2000 images, 3 epochs).


In [2]:
%run scripts/train_detector.py --mode train --config configs/config.yaml --limit-images 200 --epochs 1 --batch-size 2 --lr 0.005 --checkpoint-dir outputs/checkpoints/smoke --log-file outputs/metrics/smoke_log.csv


ValueError: Categories not found in COCO annotations: ['bus', 'dog', 'elephant', 'umbrella', 'keyboard']

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("outputs/metrics/smoke_log.csv")
cols = [c for c in log.columns if c.startswith("loss_")]
log[cols].plot(figsize=(9, 4))
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("Smoke-run losses per epoch")
plt.tight_layout()
plt.show()
print(log.tail(3).to_string(index=False))


When the smoke run shows finite, decreasing losses, run full training:

```bash
python scripts/train_detector.py --mode train --config configs/config.yaml \
    --epochs 3 --batch-size 2 --lr 0.005 --checkpoint-dir outputs/checkpoints
```

On Windows CPU this takes roughly 1–3 hours. Checkpoints save every epoch to
`outputs/checkpoints/` and are resumable if the machine is interrupted.
